In [1]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import pytorch_lightning as pl
from transformers import DataCollatorForSeq2Seq, AutoTokenizer
from torch.utils.data import DataLoader
import os
import pickle
from datasets import load_dataset

/Users/pavlospoulos/miniconda3/envs/pavlosEnv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# English to Romanian
dataset = load_dataset("IWSLT/iwslt2017", "iwslt2017-en-ro", trust_remote_code=True)

Generating validation split: 100%|██████████| 914/914 [00:00<00:00, 63947.59 examples/s]


In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 220538
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 1678
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 914
    })
})

In [9]:
dataset['train']['translation'][0]

{'en': "Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.",
 'ro': 'Mulțumesc foarte mult, Chris. Și trebuie să spun că mă simt onorat să am ocazia de a veni pe scenă de două ori. Sunt foarte recunoscător.'}

In [11]:
dataset['test']['translation'][0]

{'en': 'Several years ago here at TED, Peter Skillman introduced a design challenge called the marshmallow challenge.',
 'ro': 'Acum câțiva ani în urmă, aici la TED, Peter Skillman  a prezentat o problemă de design  numită problema bezelei.'}

In [12]:
dataset['validation']['translation'][0]

{'en': 'Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.',
 'ro': 'Anul trecut am aratat aceste doua diapozitive pentru a  demonstra ca intreaga calota polara  care in marea majoritate a ultimilor 3 milioane de ani  a fost de dimensiunea a 48 de state de marime mica,  s-a micsorat cu 40% .'}

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google-t5/t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [14]:
model.config.task_specific_params

{'summarization': {'early_stopping': True,
  'length_penalty': 2.0,
  'max_length': 200,
  'min_length': 30,
  'no_repeat_ngram_size': 3,
  'num_beams': 4,
  'prefix': 'summarize: '},
 'translation_en_to_de': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to German: '},
 'translation_en_to_fr': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to French: '},
 'translation_en_to_ro': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to Romanian: '}}

# Testing

In [62]:
import os
import pickle
import pytorch_lightning as pl
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

class T5TranslationDataModule(pl.LightningDataModule):
    def __init__(self, model_name, dataset_name, max_length, 
                 batch_size, train_range, val_range, test_range, seed_num):
        super().__init__()
        self.model_name = model_name
        self.dataset_name = dataset_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.train_range = train_range
        self.val_range = val_range 
        self.test_range = test_range
        self.seed_num = seed_num
        self.tokenizer = None
        self.data_collator = None
        self.train_datasets = []
        self.val_datasets = []
        self.test_datasets = []
        self.cache_dir = f"./dataset_cache_{self.seed_num}"
        self.datasets = {}
        # Load datasets for each language pair
        self.language_codes = {
            "Romanian": "iwslt2017-en-ro",
            "German": "iwslt2017-en-de",
            "French": "iwslt2017-en-fr"
        }

    def prepare_data(self):
        for lang in self.language_codes.keys():
            code = self.language_codes[lang]
            self.datasets[lang] = load_dataset(self.dataset_name, code, trust_remote_code=True)
            self.datasets[lang] = self.datasets[lang].shuffle(seed=self.seed_num)

        # Download tokenizer
        AutoTokenizer.from_pretrained(self.model_name)

    def setup(self, stage=None):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.data_collator = DataCollatorForSeq2Seq(tokenizer=self.tokenizer, model=self.model_name)
        
        if stage == 'fit' or stage is None:
            self.train_datasets = self._get_or_process_dataset('train')
            self.val_datasets = self._get_or_process_dataset('validation')
        if stage == 'test' or stage is None:
            self.test_datasets = self._get_or_process_dataset('test')
        
        print(f"Setup complete. Datasets sizes: Train: {len(self.train_datasets)}, Val: {len(self.val_datasets)}, Test: {len(self.test_datasets)}")

    def _get_or_process_dataset(self, split):
        combined_dataset = []
        
        for language in self.language_codes.keys():
            cache_file = os.path.join(self.cache_dir, f"{split}_{language}_{self.seed_num}.pkl")
            
            if os.path.exists(cache_file):
                print(f"Loading cached {split} dataset for {language}...")
                with open(cache_file, 'rb') as f:
                    dataset = pickle.load(f)
            else:
                print(f"Processing {split} dataset for {language}...")                
                if split == 'train':
                    train_dataset = self.datasets[language]['train']
                    dataset = train_dataset.select(range(min(self.train_range, len(train_dataset))))
                elif split == 'validation':
                    val_dataset = self.datasets[language]['validation']
                    dataset = val_dataset.select(range(min(self.val_range, len(val_dataset))))
                elif split == 'test':
                    test_dataset = self.datasets[language]['test']
                    dataset = test_dataset.select(range(min(self.test_range, len(test_dataset))))
                
                processed_dataset = self._preprocess_dataset(dataset, language)
                
                os.makedirs(self.cache_dir, exist_ok=True)
                with open(cache_file, 'wb') as f:
                    pickle.dump(processed_dataset, f)
                
                dataset = processed_dataset
            
            print(f"Loaded {split} dataset for {language} with {len(dataset)} samples")
            combined_dataset.extend(dataset)
        
        return combined_dataset
    
    def _preprocess_dataset(self, dataset, target_language):
        def preprocess_function(examples):
            model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
            second_mapping = {
                "Romanian": "ro",
                "German": "de",
                "French": "fr"
            }

            for i in range(len(examples['translation'])):
                prefix = f"translate English to {target_language.capitalize()}: "
                input_text = prefix + examples['translation'][i]['en']
                target_text = examples['translation'][i][second_mapping[target_language]]
                
                tokenized_input = self.tokenizer(input_text, max_length=self.max_length, padding="max_length", truncation=True)
                tokenized_target = self.tokenizer(target_text, max_length=self.max_length, padding="max_length", truncation=True)
                
                model_inputs["input_ids"].append(tokenized_input["input_ids"])
                model_inputs["attention_mask"].append(tokenized_input["attention_mask"])
                model_inputs["labels"].append(tokenized_target["input_ids"])

            return model_inputs

        return dataset.map(
            preprocess_function,
            batched=True,
            remove_columns=dataset.column_names
        )

    def train_dataloader(self):
        return DataLoader(self.train_datasets, batch_size=self.batch_size, collate_fn=self.data_collator, shuffle=True, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_datasets, batch_size=self.batch_size, collate_fn=self.data_collator, drop_last=True)

    def test_dataloader(self):
        return DataLoader(self.test_datasets, batch_size=self.batch_size, collate_fn=self.data_collator, drop_last=True)

In [63]:
model_name = "t5-small"  # or your specific model name
dataset_name = "IWSLT/iwslt2017"  # or your specific dataset name
max_length = 128
batch_size = 32
train_range = 100 # adjust as needed
val_range = 100  # adjust as needed
test_range = 100  # adjust as needed
seed_num = 42

data_module = T5TranslationDataModule(
    model_name, dataset_name, max_length, batch_size,
    train_range, val_range, test_range, seed_num
)

In [64]:
data_module.prepare_data()

In [66]:
data_module.setup('train')

Setup complete. Datasets sizes: Train: 0, Val: 0, Test: 300


In [275]:
list_with_english = []
list_with_translated = []
for i in range(300):
    result_of_iterator = next(iter(data_module.train_dataloader()))
    list_with_english.append(" ".join(data_module.tokenizer.batch_decode(result_of_iterator['input_ids'][0])))
    list_with_translated.append(" ".join(data_module.tokenizer.batch_decode(result_of_iterator['labels'][0])))


In [290]:
df = pd.DataFrame({"English": list_with_english, "Translated": list_with_translated})
df['English'].apply(lambda x: x.split(" : ")[1]).unique()

array(["A hostel collapse d in Me cca , the holy city of Islam at about 10  o ' clock this morning local time . </s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>",
       'O cca s ional specialist air tours go  inland , for mountain e er ing or to reach the Pole , which has  a large base . </s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>